## Athar Sale
#### Notebook for the keyword analysis of data output from the AS_data_collection.py tool

##### the data pipeline starts with **raw listings** ('AS_output_restricted.json')
##### **Step 1:** normalisation of both the listing text (title + description) and the keywords
##### **Step 2:** keyword matching and scoring of the listings according to the frequency of matched keywords
##### **Step 3:** implement score threshold and filter accordingly, to classify listings into high-, medium-, and low-risk
##### **Step 4:** manual review and validation/adjustments to keywords if necessary

In [43]:
import re
import glob
import json
import unicodedata
import unicodedata2
import collections
from collections import Counter

In [44]:
# first we want to prepare the data for analysis; start by importing the previously output restricted dataset (available in GitHub)
# Athar Sale has introduced a captcha mechanism on their website so we will cease automated collection of new data
objs_file = glob.glob('AS_output_restricted.json')[0]
with open(objs_file, 'r', encoding="utf-8") as data:
    data = json.load(data)

# using this data (5,917 listings), create a new list of dictionaries comprising just listing IDs and the text to be analysed (titles + descriptions)
new_data = []
for x in data:
    iden = x['listing_identifier']
    text = f"{x['title']} | {x['description']}" if x['description'] != '' else x['title']   # concatenate title and description if desc isn't empty
    new_data.append({'listing_id': iden,
                     'listing_text': text})

In [45]:
# next we want to normalise the Arabic text to remove accents and diacritics, letter variants, numbers, punctuation, etc
ARABIC_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")

def normalise_arabic(text: str) -> str:
    """
    normalise Arabic text to enable keyword matching:
    - removes tatweel, diacritics (tashkeel), punctuation, and extra spaces
    - normalises alifs, yaa, taa marbouta, and hamza
    """
    if not isinstance(text, str):
        return text

    text = unicodedata.normalize("NFC", text)           # unicode normalise
    text = text.replace("ـ", "")                         # remove tatweel
    text = ARABIC_DIACRITICS.sub("", text)              # remove diacritics
    text = re.sub(r"[إأآٱ]", "ا", text)                    # normalise alif variants
    text = re.sub(r"ى", "ي", text)                      # normalise yaa/alif muksoura
    text = re.sub(r"ة", "ه", text)                       # normalise taa marbouta
    text = re.sub(r"[ؤئ]", "ء", text)                   # normalisa hamza variants
    text = text.replace("\n", " | ")                    # preserve structure by replacing new line breaks with | separator
    # text = re.sub(r"[|/\\\-•]", " | ", text)
    text = re.sub(r"\d+", " ", text)                    # remove numbers
    text = re.sub(r"[^\w\s|]", " ", text)               # remove remaining punctuation but keep | separators
    text = re.sub(r"\s+", " ", text).strip()            # remove whitespace
    text = text.lower()

    return text

for x in new_data:
    x['normalised_text'] = normalise_arabic(x['listing_text'])

In [97]:
# import relevant keywords
with open("keywords.json", "r", encoding="utf-8") as kf:
    KEYWORDS = json.load(kf)

# import negative keywords (to eliminate false positives)
with open("negative_keywords.json", "r", encoding="utf-8") as nk:
    NEGATIVES = json.load(nk)

# apply the normalise_arabic function to the keywords lists to maximise likelihood of matches in the text
for element in KEYWORDS:
    new_list = []
    for s in KEYWORDS[element]['terms']:
        new_list.append(normalise_arabic(s))
    KEYWORDS[element]['terms'] = new_list
for element in NEGATIVES:
    NEGATIVES[NEGATIVES.index(element)] = normalise_arabic(element)

In [98]:
# KEYWORD MATCHING
# flatten keywords into term -> weight
def flatten_keywords(keyword_dict, default_weight=1):
    flat = {}
    for group, data in keyword_dict.items():
        weight = data.get("weight", default_weight)
        for term in data["terms"]:
            flat[term.lower()] = weight
    return flat

# flatten negative keywords list (words and phrases which automatically indicate this is an irrelevant listing)
def flatten_negatives(neg_list):
    return [n.lower() for n in neg_list]

# function to check if text contains negative keywords
def contains_negative(tokens, negatives):
    token_set = set(tokens)
    for n in negatives:
        n_tokens = n.split()
        if len(n_tokens) == 1:
            if n_tokens[0] in token_set:
                return True
        else:
            for i in range(len(tokens) - len(n_tokens) + 1):
                if tokens[i:i+len(n_tokens)] == n_tokens:
                    return True
    return False
    
# safe tokenisation (fixed) - removes tokens like "L", "D", "3"
def tokenize(text):
    tokens = re.findall(r"[a-zA-Z\u0600-\u06FF]+", text.lower())
    cleaned = []
    for t in tokens:
        if len(t) < 2:                                                       # removes single-letter tokens (L, D, a, etc.)
            continue
        if len(t) == 2 and t in {"ur", "al", "el", "in", "ar", "an"}:        # removes short tokens because of risk of false positives
            continue
        cleaned.append(t)
    return cleaned

# exact phrase matching (token-based)
def phrase_match(term_tokens, tokens):
    n = len(term_tokens)
    for i in range(len(tokens) - n + 1):
        if tokens[i:i+n] == term_tokens:
            return True
    return False

# function to score listings according to keyword matches
def score_text(listing, keyword_dict):
    text = listing['normalised_text'].lower()
    tokens = tokenize(text)

    # exclude listings which contain keywords from the NEGATIVES list
    negatives_flat = flatten_negatives(NEGATIVES)
    if contains_negative(tokens, negatives_flat):
        listing['score'] = 0
        listing['matches'] = {"NEGATIVE_FLAG"}
        return listing
    
    if not tokens:
        listing['score'] = 0
        listing['matches'] = set()
        return listing
    token_set = set(tokens)
    flat_keywords = flatten_keywords(keyword_dict)
    total_score = 0
    matched_terms = set()
    for term, weight in flat_keywords.items():
        term_norm = term.lower()
        if term_norm in matched_terms:
            continue
        term_tokens = tokenize(term_norm)
        if len(term_tokens) == 1:                                            # single-word keywords
            if term_tokens[0] in token_set:
                total_score += weight
                matched_terms.add(term_norm)
        else:                                                                # multiple-word keywords (phrases)
            if phrase_match(term_tokens, tokens):
                total_score += weight
                matched_terms.add(term_norm)

    # add score and keyword matches to the listing dictionary
    listing['score'] = total_score
    listing['matches'] = matched_terms

    return listing

# apply the function to the list of listing dictionaries
for x in new_data:
    score_text(x, KEYWORDS)

In [105]:
# sort the listings in reverse order by score, for manual examination/validation
sorted_listings = sorted(new_data, key=lambda x: x["score"], reverse=True)

sorted_listings

[{'listing_id': 'AS_847407819851050',
  'listing_text': 'اثار عراقيه اشوريه بابلى لاهل بابل بالعراق اصلي حجر تقيل مقاس 44* 33 سنتي | اثار عراقيه اشوريه بابلى لاهل بابل بالعراق اصلي حجر تقيل مقاس 44* 33 سنتي عمرها اكثر من 3000 سنه لنابوخذنصر ',
  'normalised_text': 'اثار عراقيه اشوريه بابلي لاهل بابل بالعراق اصلي حجر تقيل مقاس سنتي | اثار عراقيه اشوريه بابلي لاهل بابل بالعراق اصلي حجر تقيل مقاس سنتي عمرها اكثر من سنه لنابوخذنصر',
  'score': 48,
  'matches': {'اشوريه', 'بابل', 'بابلي', 'حجر', 'عراقيه', 'لنابوخذنصر'}},
 {'listing_id': 'AS_303990590309595',
  'listing_text': 'Rare and Scarce Iraqi Coin 50 Fils | Rare and Scarce Iraqi Coin 50 Fils, 1955 Issue\nMade of silver from the Kingdom of Iraq during the reign of King Faisal II\nFor sale to the highest bidder',
  'normalised_text': 'rare and scarce iraqi coin fils | rare and scarce iraqi coin fils issue | made of silver from the kingdom of iraq during the reign of king faisal ii | for sale to the highest bidder',
  'score': 30,
  'mat

In [106]:
# thresholds have been refined after manual analysis of highest scoring listings
top_threshold = 12
bottom_threshold = 8

# categorise listings based on score
high_priority = [l for l in sorted_listings if l["score"] >= top_threshold]
medium_priority = [l for l in sorted_listings if l["score"] < top_threshold and l["score"] > bottom_threshold]
low_priority = [l for l in sorted_listings if l["score"] <= bottom_threshold and l["score"] > 0]
unrelated = [l for l in sorted_listings if l["score"] == 0]

# create lists of the listing IDs which we will use to retrieve the URLs and other identifying information from the unrestricted dataset
high_priority_listings = [x['listing_id'] for x in high_priority]
medium_priority_listings = [x['listing_id'] for x in medium_priority]

print(f"there are {len(high_priority)} listings which almost certainly depict Iraqi cultural goods")
print(f"there are {len(medium_priority)} listings which possibly depict Iraqi cultural goods")
print(f"there are {len(low_priority)} listings which likely do not depict Iraqi cultural goods")
print(f"there are {len(unrelated)} listings which do not contain any keyword matches")

there are 71 listings which almost certainly depict Iraqi cultural goods
there are 72 listings which possibly depict Iraqi cultural goods
there are 3222 listings which likely do not depict Iraqi cultural goods
there are 2552 listings which do not contain any keyword matches


In [107]:
# import original unrestricted data
# note: the unrestricted dataset is not publicly available because it contains personal identifying information and active links
og_data = glob.glob('AS_output_full.json')[0]
with open(og_data, 'r', encoding="utf-8") as data:
    objs = json.load(data)

# use listing ID lists to identify high and medium risk listings
high_risk = [x for x in objs if "AS_" + x['obj_uuid'].partition("_")[2] in high_priority_listings]
med_risk = [x for x in objs if "AS_" + x['obj_uuid'].partition("_")[2] in medium_priority_listings]

In [102]:
# determine the frequency of high-risk listings by country; this needs to be tallied manually because there's no standard way to identify countries
print(len(high_risk))
for x in high_risk:
    print(f"{high_risk.index(x)}: {x['seller_country']}")

# 27 Egypt
# 13 Saudi Arabia
# 5 Ethiopia
# 5 Iraq
# 4 Syria
# 4 United Arab Emirates
# 3 Jordan
# 2 Austria
# 2 Lebanon
# 2 Morocco
# 1 Algeria
# 1 Denmark
# 1 Germany
# 1 Palestine

71
0: الإمارات
1: العراق
2: الأردن
3: المملكة العربية السعودية
4: لبنان
5: المغرب
6: الجزائر
7: United Arab Emirates
8: ألمانيا
9: مصر
10: مصر
11: مصر
12: الدنمارك
13: Egypt
14: الإمارات
15: سوريا
16: المملكة العربية السورية  -  جدة
17: مصر
18: مصر
19: مصر
20: مصر
21: مصر
22: مصر
23: المملكة العربية السعودية
24: مصر
25: مصر
26: سوريا
27: مصر
28: مصر
29: مصر
30: فلسطين الداخل
31: مصر
32: مصر
33: المملكة العربية السعودية
34: مصر
35: إثيوبيا - Ethiopia
36: إثيوبيا - Ethiopia
37: إثيوبيا - Ethiopia
38: إثيوبيا
39: إثيوبيا - Ethiopia
40: Egypt
41: مصر
42: مصر
43: Saudi Arabia
44: Saudi Arabia
45: Saudi Arabia
46: Saudi Arabia
47: مصر
48: مصر
49: العراق Iraq
50: المغرب
51: مصر
52: الأردن
53: المملكة العربية السعوديه
54: المملكه العربية السعودية  ينبع
55: النمسا
56: المملكة العربية السعودية
57: العراق
58: سوريا
59: الأردن
60: مصر
61: United Arab Emirates
62: العراق
63: النمسا
64: لبنان
65: العراق
66: مصر
67: السعوديه
68: مصر
69: سوريا
70: المملكه العربيه السعوديه


In [103]:
# use the same manual methodology for medium-risk listings to determine frequency by country
print(len(med_risk))
for x in med_risk:
    print(f"{med_risk.index(x)}: {x['seller_country']}")

# 28 Egypt
# 17 Saudi Arabia
# 4 Iraq
# 3 Syria
# 3 Jordan
# 3 Morocco
# 3 Yemen
# 2 Lebanon
# 1 United Arab Emirates
# 1 Algeria
# 1 Denmark
# 1 Finland
# 1 Germany
# 1 Kuwait
# 1 Libya
# 1 Sweden
# 1 Turkey

72
0: الأردن
1: السعودية
2: المملكة المغربية
3: لبنان
4: اليمن
5: سوريا
6: syria
7: مصر
8: Egypt
9: مصر  القاهرة
10: تركيا
11: ألمانيا
12: اليمن
13: مصر
14: مصر
15: العراق
16: سوريا
17: مصر
18: القاهرة
19: الكويت
20: المغرب
21: ليبيا
22: الأردن
23: Algérie
24: السعودية
25: Lebanon
26: مصر
27: مصر
28: مصر
29: المملكة العربية السعودية
30: الأردن
31: السعودة
32: مصر
33: مصر
34: Egypt
35: المملكة العربية السعودية
36: مصر
37: مصر
38: مصر
39: اليمن
40: المغرب
41: المملكة العربية السعودية
42: السويد ستكهولم
43: مصر
44: السعودية
45: Egypt
46: مصر
47: مصر
48: Egypt
49: مصر
50: السعودية
51: مصر
52: العراق
53: Saudi Arabia
54: مًصّر
55: United Arab Emirates
56: فنلندا
57: المملكة العربية السعودية
58: العراق
59: العراق(Iraq)
60: السعودية
61: السعوديه
62: مصر
63: مصر
64: المملكة العربية السعودية
65: المملكه العربية السعودية  Saudi Arabia
66: السعوديه
67: مصر
68: الدنمارك
69: السعودية
70: السعودية
71: مصر


In [133]:
# modify the formatting of the listing IDs so they can be parsed correctly
for x in high_risk:
    x['obj_uuid'] = f"AS_{x['obj_uuid'].partition("_")[2]}"
for x in med_risk:
    x['obj_uuid'] = f"AS_{x['obj_uuid'].partition("_")[2]}"

# use the list to remove Iraq-based sellers from the high- and medium-risk lists, so we can focus solely on sellers abroad
new_high_risk = [x for x in high_risk if "عراق" not in x['seller_country']]
new_med_risk = [x for x in med_risk if "عراق" not in x['seller_country']]

high_risk_uuids = [x['obj_uuid'] for x in new_high_risk]
med_risk_uuids = [x['obj_uuid'] for x in new_med_risk]

# do the same for the dicts with the keyword match scores, so we can analyse frequency of keywords in listings by sellers outside of Iraq
new_high_priority = [x for x in high_priority if x['listing_id'] in high_risk_uuids]
new_medium_priority = [x for x in medium_priority if x['listing_id'] in med_risk_uuids]

print(len(new_high_risk))         # 71 -> 66
print(len(new_med_risk))          # 72 -> 68

66
68


In [134]:
# determine how many of the high- and medium-risk listings include a price
print(len([x for x in high_risk if x['price'] != '']) + len([x for x in med_risk if x['price'] != '']))
# 33.83% of listings have a price

51


In [135]:
# print all prices to manually determine range (due to inconsistent formatting and multiple currencies)
for y in [x['price'] for x in high_risk if x['price'] != '']:
    print(y)
for y in [x['price'] for x in med_risk if x['price'] != '']:
    print(y)

## TOTALS
# USD 11951
# saudi riyal 27100 ($7,224.55 USD)
# egypt pound 4300 ($87.53 USD)
# JOD 5 ($7.05 USD)
# emirates dinar 1 ($0.27 USD)

# highest price - 20,000 saudi riyals ($5,337.18 USD)
# lowest - 1 emirati dinar ($0.27 USD)

45 ريال سعودي
$200
$35
$1
$1
20,000 ريال سعودي
$50
$20
500 جنيه مصري
$1
$10
$30
$10
$25
$30
$10
$15
150 جنيه مصري
$499
$200
1 د.إماراتي
$1
$1,900
$2
2,500 جنيه مصري
$85
25 ريال سعودي
$1,000
$3,000
$200
$11
$111
$250
$3,000
5 د.أردني
150 جنيه مصري
$20
$200
$10
$10
30 ريال سعودي
$33
$75
$300
$1
$50
5,500 ريال سعودي
1,500 ريال سعودي
$350
$200
1,000 جنيه مصري


In [136]:
# determine how many unique sellers there are for the high- and medium-risk listings
sellers1 = [x['seller_name'] for x in high_risk]
sellers1.extend([x['seller_name'] for x in med_risk])

sellers1 = list(set(sellers1))
len(sellers1)

105

In [138]:
# determine the frequency of each matched keyword in the high-priority listings
hp_matches = []
for x in high_priority:
    for element in x['matches']:
        hp_matches.append(element)

counts = Counter(hp_matches)
counts

## I then manually merged counts of EN/AR keywords (e.g., iraq/iraqi (عراقي)) to get this list
## we have to tally them up manually, unfortunately, because there are multiple versions of each keyword that count as the same term
## actually I'm sure there's a more efficient way to do this but in the interest of time I'm just doing it manually

#  'arab/arabic (عربي)': 24,
#  'coin (عمله)': 33,
#  'iraq/iraqi (عراقي / العراق / عراقيه)': 42,
#  'old (قديم)': 20,
#  'fils/fals (فلس)': 12,
#  'islamic': 11,
#  'babylonian/babylon (بابلي/بابل)': 5,
#  'faisal (فيصل'): 5,
#  'stone (حجر)': 5,
#  'ancient': 4,
#  'assyria (اشوريه)': 3,
#  'calligraphy': 3,
#  'ghazi (غازي)': 4,
#  'gold (ذهب)': 4,
#  'quran': 4,
#  'statue (تمثال)': 3,
#  'abbasid': 2,
#  'banknote': 2,
#  'dinar (دينار)': 4,
#  'agate (عقيق)': 2,
#  'dirham (درهم)': 2,
#  'hashemite (الهاشميه)': 2,
#  'kufic/kufi (كوفي)': 2,
#  'manuscript (مخطوطه)': 2,
#  'ottoman (عثمان)': 3,
#  'sumerian (سومري)': 3,
#  'umayyad (اموي)': 3,
#  'adab (اداب)': 1,
#  'artifact': 1,
#  'atabeg': 1,
#  'baghdad (بغداد)': 2,
#  'basra (البصره)': 1,
#  'chalcedony seal (ختم عقيق)': 1,
#  'cuneiform': 1,
#  'cylinder seal': 1,
#  'dagger': 3,
#  'mamluk (مملوكي)': 1,
#  'medieval': 1,
#  'najaf (النجف)': 1,
#  'nebuchadnezzar (لنابوخذنصر)': 1,
#  'pendant (قلاده)': 1,
#  'pottery': 1,
#  'prehistoric (ما قبل التاريخ)': 1,
#  'ring (خاتم)': 2,
#  'samarra (سامراء)': 1,
#  'seljuk': 1,
#  'sword': 2,
#  'ur iii': 1,
#  'waqf (وقف)': 1,
#  'wasit (واسط)': 1,
#  'disc (قرص)': 1,
#  'necklace (عقد)': 1,
#  'envelope (غلاف)': 1,
#  'der': 1,
#  'museum': 1
# }

Counter({'عمله': 23,
         'عراقيه': 20,
         'arabic': 13,
         'old': 11,
         'coin': 10,
         'islamic': 10,
         'فلس': 9,
         'عراقي': 8,
         'arab': 8,
         'العراق': 6,
         'قديمه': 6,
         'iraqi': 5,
         'iraq': 5,
         'حجر': 4,
         'ancient': 4,
         'اشوريه': 3,
         'faisal': 3,
         'calligraphy': 3,
         'dagger': 3,
         'quran': 3,
         'gold': 3,
         'قديم': 3,
         'عربي': 3,
         'سومري': 3,
         'تمثال': 3,
         'بابلي': 2,
         'بابل': 2,
         'fils': 2,
         'abbasid': 2,
         'ottoman': 2,
         'ghazi': 2,
         'غازي': 2,
         'dinar': 2,
         'banknote': 2,
         'بغداد': 2,
         'manuscript': 2,
         'عقيق': 2,
         'درهم': 2,
         'فيصل': 2,
         'umayyad': 2,
         'دينار': 2,
         'لنابوخذنصر': 1,
         'اسلامي': 1,
         'fals': 1,
         'kufic': 1,
         'artifact': 1,
         

In [139]:
# do the same for the medium-priority listings
mp_matches = []
for x in medium_priority:
    for element in x['matches']:
        mp_matches.append(element)

counts = Counter(mp_matches)
counts

# {'old (قديم)': 44,
#  'ottoman (عثماني / عثمان)': 21,
#  'quran: (مصحف / قران)': 16,
#  'arab/arabic (عربي)': 9,
#  'coin': 14,
#  'iraq/iraqi (العراق / عراقي)': 13,
#  'gold (ذهب)': 7,
#  'dagger (خنجر)': 5,
#  'islamic': 6,
#  'manuscript (مخطوطه)': 5,
#  'sword (سيف)': 5,
#  'abbasid (عباسي / عباس)': 4,
#  'ancient (عتيق)': 5,
#  'banknote': 3,
#  'der (دير)': 3,
#  'dinar (دينار)': 3,
#  'museum': 3,
#  'necklace (عقد)': 3,
#  'dirham (درهم)': 3,
#  'faisal (فيصل)': 4,
#  'fils (فلس)': 2,
#  'medieval': 2,
#  'ring (خاتم)': 1,
#  'amorite (امور)': 1,
#  'antiquity': 1,
#  'astrolabe': 1,
#  'calligraphy': 1,
#  'hashemite (الهاشميه)': 3
#  'historic (تاريخي)': 1,
#  'mesopotamia': 1,
#  'mongol (مغولي)': 1
#  'oriental': 1,
#  'umayyad (اموي)': 2,
#  'waqf (وقف)': 1,
#  'kufic/kufi (كوفي)': 1,
#  'ur iii': 1,
#  'envelope (غلاف)': 1
# }

Counter({'قديم': 24,
         'عثماني': 15,
         'old': 14,
         'مصحف': 9,
         'عمله': 8,
         'قديمه': 6,
         'coin': 6,
         'خنجر': 5,
         'قران': 4,
         'ذهب': 4,
         'ottoman': 4,
         'islamic': 4,
         'سيف': 4,
         'مخطوطه': 3,
         'iraqi': 3,
         'الهاشميه': 3,
         'دينار': 3,
         'iraq': 3,
         'العراق': 3,
         'ancient': 3,
         'quran': 3,
         'دير': 3,
         'فيصل': 3,
         'gold': 3,
         'banknote': 3,
         'arab': 3,
         'عربي': 3,
         'درهم': 3,
         'arabic': 3,
         'museum': 2,
         'عراقي': 2,
         'عتيق': 2,
         'عباسي': 2,
         'medieval': 2,
         'manuscript': 2,
         'عقد': 2,
         'عثمان': 2,
         'فلس': 2,
         'عراقيه': 2,
         'اسلامي': 2,
         'اموي': 2,
         'mesopotamia': 1,
         'متحف': 1,
         'كوفي': 1,
         'astrolabe': 1,
         'antiquity': 1,
         'وقف': 1,

In [ ]:
high_and_medium_matches = {
#  'old (قديم)': 64,
#  'iraq/iraqi (عراقي / العراق / عراقيه)': 55,
#  'coin (عمله)': 47,
#  'arab/arabic (عربي)': 33,
#  'ottoman (عثمان)': 24,
#  'quran': 20,
#  'islamic': 17,
#  'fils/fals (فلس)': 14,
#  'gold (ذهب)': 11,
#  'ancient': 9,
#  'faisal (فيصل'): 9,
#  'dagger': 8,
#  'dinar (دينار)': 7,
#  'manuscript (مخطوطه)': 7,
#  'sword': 7,
#  'abbasid': 6,
#  'banknote': 5,
#  'babylonian/babylon (بابلي/بابل)': 5,
#  'dirham (درهم)': 5,
#  'hashemite (الهاشميه)': 5,
#  'stone (حجر)': 5,
#  'umayyad (اموي)': 5
#  'calligraphy': 4,
#  'ghazi (غازي)': 4,
#  'necklace (عقد)': 4,
#  'der (دير)': 4,
#  'museum': 4
#  'assyria (اشوريه)': 3,
#  'statue (تمثال)': 3,
#  'kufic/kufi (كوفي)': 3,
#  'ring (خاتم)': 3,
#  'sumerian (سومري)': 3,
#  'baghdad (بغداد)': 2,
#  'envelope (غلاف)': 2,
#  'agate (عقيق)': 2,
#  'medieval': 2,
#  'ur iii': 2,
#  'waqf (وقف)': 2,
#  'adab (اداب)': 1,
#  'artifact': 1,
#  'atabeg': 1,
#  'basra (البصره)': 1,
#  'chalcedony seal (ختم عقيق)': 1,
#  'cuneiform': 1,
#  'cylinder seal': 1,
#  'mamluk (مملوكي)': 1,
#  'najaf (النجف)': 1,
#  'nebuchadnezzar (لنابوخذنصر)': 1,
#  'pendant (قلاده)': 1,
#  'pottery': 1,
#  'prehistoric (ما قبل التاريخ)': 1,
#  'samarra (سامراء)': 1,
#  'seljuk': 1,
#  'wasit (واسط)': 1,
#  'disc (قرص)': 1,
#  'amorite (امور)': 1,
#  'antiquity': 1,
#  'astrolabe': 1,
#  'historic (تاريخي)': 1,
#  'mesopotamia': 1,
#  'mongol (مغولي)': 1
#  'oriental': 1,
    
cultural_historical = {
#  'iraq/iraqi (عراقي / العراق / عراقيه)': 55,
#  'arab/arabic (عربي)': 33,
#  'ottoman (عثمان)': 24,
#  'islamic': 17,
#  'faisal (فيصل'): 9,
#  'ancient': 9,
#  'abbasid': 6,
#  'babylonian/babylon (بابلي/بابل)': 5,
#  'hashemite (الهاشميه)': 5,
#  'umayyad (اموي)': 5
#  'ghazi (غازي)': 4,
#  'assyria (اشوريه)': 3,
#  'sumerian (سومري)': 3,
#  'ur iii': 2,
#  'medieval': 2,
#  'atabeg': 1,
#  'mamluk (مملوكي)': 1,
#  'nebuchadnezzar (لنابوخذنصر)': 1,
#  'prehistoric (ما قبل التاريخ)': 1,
#  'seljuk': 1,
#  'amorite (امور)': 1,
#  'mesopotamia': 1,
#  'mongol (مغولي)': 1

physical_locations = [
#  'iraq/iraqi (عراقي / العراق / عراقيه)': 55,
#  'babylonian/babylon (بابلي/بابل)': 5,
#  'der (دير)': 4,
#  'assyria (اشوريه)': 3,
#  'baghdad (بغداد)': 2,
#  'adab (اداب)': 1,
#  'basra (البصره)': 1,
#  'najaf (النجف)': 1,
#  'samarra (سامراء)': 1,
#  'wasit (واسط)': 1,
#  'mesopotamia': 1

object_types = {
#  'coin (عمله)': 47,
#  'quran': 20,
#  'fils/fals (فلس)': 14,
#  'gold (ذهب)': 11,
#  'dagger': 8,
#  'dinar (دينار)': 7,
#  'manuscript (مخطوطه)': 7,
#  'sword': 7,
#  'banknote': 5,
#  'dirham (درهم)': 5,
#  'stone (حجر)': 5,
#  'necklace (عقد)': 4,
#  'statue (تمثال)': 3,
#  'ring (خاتم)': 3,
#  'envelope (غلاف)': 2,
#  'agate (عقيق)': 2,
#  'chalcedony seal (ختم عقيق)': 1,
#  'cuneiform': 1,
#  'cylinder seal': 1,
#  'pendant (قلاده)': 1,
#  'pottery': 1,
#  'disc (قرص)': 1,
#  'astrolabe': 1

In [40]:
# we developed these smaller lists of site-specific keywords so we can filter listings for those related specifically to one of the four Iraqi sites
# on the Athar Sale Archaeological Map
babylon = [
    "Babylon", "Babilon", "Babil", 
    "بابل", "مدينة بابل", "آثار بابل", "بابل القديمة"
]

tell_harmal = [
    "Tell Harmal", "Tal Harmal", "Til Harmal", "Tell Harmel", "Shaduppum", "Shadupum", 
    "تل حرمل", "شادوبوم", "حرمل", "تل الحرمل"
]

aqar_quf = [
    "Aqar Quf", "Aqarquf", "Agar Quf", "Aqar-Quf", "Dur-Kurigalzu", "Dur Kurigalzu", 
    "عقرقوف", "عقر قوف", "دور كوريغالزو", "زقورة عقرقوف"
]

kirkuk_citadel = [
    "Kirkuk Citadel", "Citadel of Kirkuk", "Kirkuk Castle", "Qalat Kirkuk", "Qal'at Kirkuk", 
    "قلعة كركوك", "القلعة في كركوك", "اثار كركوك", "قلعه كركوك"
]

site_keywords = [babylon, tell_harmal, aqar_quf, kirkuk_citadel]

# normalise Arabic letters, variants, diacritics, etc.
for l in site_keywords:
    for s in l:
        l[l.index(s)] = normalise_arabic(s)

In [41]:
# retrive listings which contain keywords from the site keyword lists
search_registry = {
    "Babylon": [k.lower() for k in babylon],
    "Tell Harmal": [k.lower() for k in tell_harmal],
    "Aqar Quf": [k.lower() for k in aqar_quf],
    "Kirkuk Citadel": [k.lower() for k in kirkuk_citadel]
}

results_by_site = {site: [] for site in search_registry}

for item in new_data:
    normalised_content = item["normalised_text"].lower()
    
    for site_name, keywords in search_registry.items():
        if any(keyword in normalised_content for keyword in keywords):
            results_by_site[site_name].append(item)

print(results_by_site)

{'Babylon': [{'listing_id': 'AS_847407819851050',
   'listing_text': 'اثار عراقيه اشوريه بابلى لاهل بابل بالعراق اصلي حجر تقيل مقاس 44* 33 سنتي | اثار عراقيه اشوريه بابلى لاهل بابل بالعراق اصلي حجر تقيل مقاس 44* 33 سنتي عمرها اكثر من 3000 سنه لنابوخذنصر ',
   'normalised_text': 'اثار عراقيه اشوريه بابلي لاهل بابل بالعراق اصلي حجر تقيل مقاس سنتي | اثار عراقيه اشوريه بابلي لاهل بابل بالعراق اصلي حجر تقيل مقاس سنتي عمرها اكثر من سنه لنابوخذنصر',
   'score': 46,
   'matches': {'اشوريه', 'بابل', 'بابلي', 'عراقيه', 'لنابوخذنصر'}},
  {'listing_id': 'AS_147305894083716',
   'listing_text': 'كتاب التلمود البابلي السموري طبعة قديمة جدا | كتاب التلمود البابلي السموري طبعة قديمة جدا',
   'normalised_text': 'كتاب التلمود البابلي السموري طبعه قديمه جدا | كتاب التلمود البابلي السموري طبعه قديمه جدا',
   'score': 0,
   'matches': set()},
  {'listing_id': 'AS_105818653196949',
   'listing_text': "de droit d'auteur pablo picasso | هاذا حقوق مؤلف لي بابلو بيكاسو موجود منها ",
   'normalised_text': 'de dr